# Interconnection Financing Triage

This notebook walks through a complete analysis of EU cross-border electricity interconnection projects. The goal is to classify each project into one of three **financing tracks** and estimate the aggregate financing needs per track, supporting a policy brief on differentiated EU financing instruments.

**What this notebook does, step by step:**

1. **Check data sources** — verify which local workbooks and manual reference tables are available
2. **Build the project master table** — load TYNDP 2024/2022 workbooks, aggregate CAPEX, merge CBA welfare values, attach credit and price data
3. **Calculate financing metrics** — annualized CAPEX, congestion-rent estimates, commercial viability ratios, social BCR, credit-constraint scores
4. **Classify projects** — assign each project to Track 1 (market-financed), Track 2 (regulated + CBCA), or Track 3 (credit-constrained)
5. **Visualize results** — scatter plot of commercial viability vs. credit constraint, stacked bar of aggregate financing needs
6. **Run sensitivity analysis** — vary discount rate, utilization factor, and credit thresholds
7. **Export** — write project-level tables, aggregate summaries, and chart files for the policy brief

**Key formulas** (from `docs/quantitative-assessment.md`):

| Metric | Formula |
|---|---|
| Capital Recovery Factor (CRF) | `r / (1 - (1+r)^(-n))` with r=5%, n=40 years |
| Annualized CAPEX | `CAPEX x CRF` |
| Estimated congestion rent | `price_diff x capacity x utilization x 8760 / 1e6` |
| Commercial viability ratio | `congestion_rent / annualized_capex` |
| Social BCR | `DSEW / annualized_capex` |
| Credit constraint score | `project_capex_share / TSO_RAB` |

**Classification logic:** if commercial ratio > 0.7 then Track 1; else if not credit-constrained then Track 2; else Track 3.

## 0. Setup

Import the `grid_financing` package and configure pandas display. All business logic lives in Python modules under `src/grid_financing/` — this notebook only orchestrates and displays results.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from grid_financing import (
    build_aggregate_summary,
    build_project_master_table,
    build_sensitivity_cases,
    calculate_project_metrics,
    classify_projects,
    export_outputs,
    manual_source_status,
)
from grid_financing.exports import build_triage_scatter_dataset, build_financing_stack_dataset
from grid_financing.classification import TRACK_1, TRACK_2, TRACK_3

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Check data sources

Before loading anything, we inspect which data files are present. The source registry knows about every required input — local TYNDP workbooks, the hourly price dataset, and the manual CSV reference tables. Each source is classified as `local` (should already be in the repo) or `manual-download-required` (you need to compile and place it in `data/manual/`).

Sources marked `exists=False` will be handled gracefully: the pipeline continues but flags affected projects with data-quality issues instead of crashing.

In [ ]:
source_status = pd.DataFrame(manual_source_status())
source_status

## 2. Build the project master table

This is the core data assembly step. `build_project_master_table()` does the following under the hood:

1. **Load `Trans.Projects`** from the TYNDP 2024 workbook — filters to the 100 cross-border transmission projects using the `Is the project cross-border?` field
2. **Load `Trans.Investments`** and aggregate `Estimated CAPEX (MEUR)` to project level by summing investment-level rows per project ID
3. **Merge EU27 CBA scenario values** from the 2024 workbook tabs (`2030NT-EU27`, `2040NT-EU27`, `2040DE-EU27`) and the 2022 workbook as fallback
4. **Apply manual overrides** from `data/manual/project_overrides.csv` (if present) — fills in missing transfer capacities and country mappings
5. **Attach participant data** from `data/manual/project_participants.csv` (if present) — for multi-country CAPEX splits
6. **Attach credit reference data** from `data/manual/tso_credit_reference.csv` (if present) — TSO ratings, RAB, sovereign data, cohesion status
7. **Build price metrics** — in development mode, uses the local hourly country-level price dataset as a proxy for cross-border price spreads
8. **Flag data-quality issues** — missing transfer capacity, missing CAPEX, missing credit data, unresolved multi-country projects, missing price inputs

The result is a single DataFrame with one row per cross-border project and all inputs needed for downstream calculations.

In [ ]:
master = build_project_master_table(development_mode=True)

print(f"Cross-border projects loaded: {len(master)}")
print(f"Projects with data-quality flags: {master['has_data_quality_issue'].sum()}")
print(f"Price input mode: {master['price_input_mode'].iloc[0]}")
print()

master[[
    "project_id", "project_name", "countries", "country_count",
    "capex_meur", "capacity_mw", "status",
    "avg_price_diff_eur_per_mwh", "data_quality_flags",
]].head(15)

## 3. Calculate financing metrics

`calculate_project_metrics()` computes the core ratios for each project:

- **Annualized CAPEX** = CAPEX x CRF (default: 5% discount rate, 40-year lifetime, CRF ~ 0.0583)
- **Estimated annual congestion rent** = avg price differential x capacity x utilization factor x 8760 hours / 1M (default utilization: 60%)
- **Commercial viability ratio** = congestion rent / annualized CAPEX. Projects above 0.7 are candidates for market financing.
- **Social BCR** = DSEW (socio-economic welfare gain) / annualized CAPEX. Used for prioritization within tracks, not for classification.
- **Credit constraint score** = project CAPEX share / TSO RAB, computed per side. The binding (higher) score determines whether the project is credit-constrained (threshold: 0.15).

In [ ]:
metrics = calculate_project_metrics(master)

metrics[[
    "project_id", "project_name",
    "annualized_capex_meur_per_year",
    "estimated_congestion_rent_meur_per_year",
    "commercial_ratio", "social_bcr",
    "credit_constraint_score", "credit_constrained",
]].head(15)

## 4. Classify projects into financing tracks

`classify_projects()` assigns each project to one of three tracks based on the metrics above:

- **Track 1 — Market-financed (merchant/hybrid):** commercial viability ratio > 0.7. These projects can plausibly cover their costs through congestion rents and do not need EU grants.
- **Track 2 — Regulated + CBCA:** not commercially viable but the sponsoring TSOs are creditworthy enough to finance via regulated tariffs with a proper Cross-Border Cost Allocation.
- **Track 3 — Credit-constrained:** not commercially viable AND the TSO(s) on at least one side are credit-constrained (score > 0.15, sub-investment-grade, or fiscally constrained sovereign). These need targeted EU support (CEF grants, EIB loans).
- **Unclassified:** projects with insufficient data to compute the required ratios (e.g. missing price inputs or transfer capacity).

After classification, the financing stack is estimated per project: how much comes from CEF grants, EIB loans, private capital, and TSO balance sheets. Track 3 CEF grant defaults are 85% for cohesion-country binding sides and 50% otherwise.

In [ ]:
classified = classify_projects(metrics)

print("Track distribution:")
print(classified["financing_track"].value_counts().to_string())
print()

classified[[
    "project_id", "project_name", "countries",
    "capex_meur", "commercial_ratio", "social_bcr",
    "credit_constraint_score", "financing_track",
    "estimated_cef_grant_meur", "data_quality_flags",
]].head(15)

## 5. Visualize: Financing Triage Scatter

This is the centrepiece chart for the policy brief. Each dot is a cross-border project:

- **X-axis:** commercial viability ratio (congestion rent / annualized CAPEX). Projects to the right of the vertical line at 0.7 are market-financeable.
- **Y-axis:** credit constraint score (project CAPEX share / TSO RAB). Projects above the horizontal line at 0.15 are credit-constrained.
- **Dot size:** total CAPEX in M EUR.
- **Colour:** financing track assignment.

The quadrants naturally map to the three tracks: top-left = Track 3 (constrained + unviable), bottom-left = Track 2 (unconstrained + unviable), right = Track 1 (commercially viable).

In [ ]:
scatter_df = build_triage_scatter_dataset(classified)

fig_scatter = px.scatter(
    scatter_df,
    x="commercial_ratio",
    y="credit_constraint_score",
    size="capex_meur",
    color="financing_track",
    hover_name="project_name",
    title="Financing Triage: Commercial Viability vs. Credit Constraint",
    labels={
        "commercial_ratio": "Commercial viability ratio",
        "credit_constraint_score": "Credit constraint score (CAPEX share / TSO RAB)",
    },
)
fig_scatter.add_vline(x=0.7, line_dash="dash", line_color="gray",
                       annotation_text="Merchant threshold (0.7)")
fig_scatter.add_hline(y=0.15, line_dash="dash", line_color="gray",
                       annotation_text="Credit constraint threshold (0.15)")
fig_scatter.update_layout(height=600)
fig_scatter.show()

## 6. Aggregate financing needs

The aggregate summary shows, per track, how many projects fall into each category and how the total CAPEX pipeline (~170 bn EUR) breaks down by financing source.

The key policy-relevant number: **by enabling merchant financing on high-congestion corridors (Track 1), a significant share of CEF resources can be redirected to the credit-constrained projects (Track 3) that need them most.**

In [ ]:
summary = build_aggregate_summary(classified)
summary[[
    "financing_track", "project_count",
    "total_capex_beur", "total_cef_beur", "total_eib_beur",
    "total_private_beur", "total_tso_beur",
]]

### Aggregate financing stack chart

Stacked bar showing the financing breakdown per track. Compare with the current CEF-E proposed budget (~17 bn EUR for the electricity share).

In [ ]:
stack_df = build_financing_stack_dataset(summary)

fig_stack = px.bar(
    stack_df,
    x="financing_track",
    y="value_beur",
    color="financing_component",
    barmode="stack",
    title="Aggregate Financing Stack by Track (bn EUR)",
    labels={"value_beur": "Financing (bn EUR)", "financing_track": ""},
    category_orders={"financing_track": [TRACK_1, TRACK_2, TRACK_3]},
)
fig_stack.update_layout(height=500)
fig_stack.show()

## 7. Sensitivity analysis

The classification thresholds and input assumptions are inherently uncertain. `build_sensitivity_cases()` re-runs the full calculation across a grid of scenarios:

- **Discount rates:** 4%, 5% (base), 6% — changes annualized CAPEX and therefore all ratios
- **Utilization factors:** 50%, 60% (base), 70% — changes congestion-rent estimates and commercial viability
- **Credit thresholds:** 10%, 15% (base), 20% — shifts projects between Track 2 and Track 3
- **Price scenarios:** any distinct scenarios in the data (currently: the local historical proxy)

This produces 3 x 3 x 3 = 27 cases per price scenario. Each case shows how sensitive the track assignments are to these assumptions.

In [ ]:
sensitivity = build_sensitivity_cases(classified)

print(f"Total sensitivity rows: {len(sensitivity)} "
      f"({sensitivity['sensitivity_case'].nunique()} cases x "
      f"{sensitivity['project_id'].nunique()} projects)")
print()

sensitivity[[
    "project_id", "project_name", "sensitivity_case",
    "commercial_ratio", "credit_constraint_score", "credit_constrained",
]].head(15)

## 8. Export

`export_outputs()` writes all results to deterministic file paths:

- `data/processed/project_master_table.csv` — full project-level table with all metrics, flags, and assumptions
- `data/processed/aggregate_summary.csv` — per-track totals
- `outputs/tables/project_level_results.xlsx` — Excel version of the project table
- `outputs/tables/aggregate_summary.xlsx` — Excel version of the summary
- `outputs/charts/triage_scatter.html` — interactive scatter plot
- `outputs/charts/aggregate_financing_stack.html` — interactive stacked bar chart

All exported tables include assumptions, source provenance, and data-quality flags so reviewers can trace any derived metric back to the workbook formulas and proxy price inputs.

In [ ]:
exports = export_outputs(classified)

print("Exported files:")
for name, path in exports.items():
    print(f"  {name}: {path}")